In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [38]:
data = pd.read_csv('Intel_CPUs.csv')
data = data[data['Vertical_Segment'].isin(['Server'])]

# Chỉ lấy các cột liên quan tới bài toán hồi quy
cols = ['Recommended_Customer_Price', 'nb_of_Cores', 'Processor_Base_Frequency', 'TDP']
data_subset = data[cols].copy()

In [39]:
# 1.1 Chuẩn hóa cột Giá (Xóa '$', ',', và chữ)
data_subset['Recommended_Customer_Price'] = data_subset['Recommended_Customer_Price'].astype(str)
data_subset['Recommended_Customer_Price'] = data_subset['Recommended_Customer_Price'].str.replace('$', '', regex=False)
data_subset['Recommended_Customer_Price'] = data_subset['Recommended_Customer_Price'].str.replace(',', '', regex=False)
data_subset['Recommended_Customer_Price'] = pd.to_numeric(data_subset['Recommended_Customer_Price'], errors='coerce')

# 1.2 Chuẩn hóa cột Xung nhịp (Đổi MHz sang GHz, xóa chữ 'GHz', 'MHz')
def clean_frequency(freq_str):
    freq_str = str(freq_str).strip()
    if pd.isna(freq_str) or freq_str == 'nan':
        return np.nan
    if 'GHz' in freq_str:
        return float(freq_str.replace(' GHz', ''))
    elif 'MHz' in freq_str:
        return float(freq_str.replace(' MHz', '')) / 1000 # Đổi MHz ra GHz
    return pd.to_numeric(freq_str, errors='coerce')

data_subset['Processor_Base_Frequency'] = data_subset['Processor_Base_Frequency'].apply(clean_frequency)

# 1.3 Chuẩn hóa cột Công suất TDP (Xóa ' W')
data_subset['TDP'] = data_subset['TDP'].astype(str).str.replace(' W', '', regex=False)
data_subset['TDP'] = pd.to_numeric(data_subset['TDP'], errors='coerce')

# 1.4 Chuẩn hóa Số lõi (Ép về numeric)
data_subset['nb_of_Cores'] = pd.to_numeric(data_subset['nb_of_Cores'], errors='coerce')

In [40]:
#Xóa hàng N/A đối với biến phụ thuộc
data_subset = data_subset.dropna(subset=['Recommended_Customer_Price'])

#Thay median cho các biến độc lập
independent_vars = ['nb_of_Cores', 'Processor_Base_Frequency', 'TDP']
for col in independent_vars:
    median_val = data_subset[col].median()
    data_subset[col] = data_subset[col].fillna(median_val)

In [41]:
def remove_outliers_iqr(data, columns):
    """ Hàm lọc bỏ ngoại lai dựa trên phương pháp Interquartile Range (IQR) """
    data_clean = data.copy()
    for col in columns:
        Q1 = data_clean[col].quantile(0.25)
        Q3 = data_clean[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Chỉ lấy nằm trong khoảng lower đến upper
        data_clean = data_clean[(data_clean[col] >= lower_bound) & (data_clean[col] <= upper_bound)]
    
    return data_clean
data_clean = remove_outliers_iqr(data_subset, cols)

In [42]:
print(data_clean.shape)

(332, 4)


In [43]:
X = data_clean[['nb_of_Cores', 'Processor_Base_Frequency', 'TDP']]
X = sm.add_constant(X)
y = data_clean['Recommended_Customer_Price']

model = sm.OLS(y, X).fit()
print("\n--- KẾT QUẢ MÔ HÌNH OLS (DESKTOP & MOBILE) ---")
print(model.summary())


--- KẾT QUẢ MÔ HÌNH OLS (DESKTOP & MOBILE) ---
                                OLS Regression Results                                
Dep. Variable:     Recommended_Customer_Price   R-squared:                       0.622
Model:                                    OLS   Adj. R-squared:                  0.619
Method:                         Least Squares   F-statistic:                     180.3
Date:                        Sun, 29 Mar 2026   Prob (F-statistic):           4.78e-69
Time:                                19:07:27   Log-Likelihood:                -2757.5
No. Observations:                         332   AIC:                             5523.
Df Residuals:                             328   BIC:                             5538.
Df Model:                                   3                                         
Covariance Type:                    nonrobust                                         
                               coef    std err          t      P>|t|      [0.025  